# Hessian-Task Vector Alignment Independence Test (IF vs Math)

This notebook estimates task independence using the second-order criterion
directly implied by TLD:

- `TLD_t = 0.5 * h_t^T H_t h_t`
- cross-task independence condition: `tau_k^T H_t tau_k ~ 0` for `k != t`

## Core Metric

For target task `t` and comparison task `k`:

- `rho(t,k) = (tau_k^T H_t tau_k) / (tau_t^T H_t tau_t)`

Interpretation:

- `rho ~ 0`: near-independent direction (flat under task-`t` curvature)
- `rho ~ 1`: cross-task curvature as large as self-task curvature

The notebook avoids full Hessian construction and uses double-autograd HVP (`H_t v`) instead.


In [ ]:
from __future__ import annotations

import gc
import json
import random
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


@dataclass(frozen=True)
class TaskSpec:
    """Configuration for one task model and its evaluation split.

    Args:
        name: Short task key used in logs/artifacts.
        model_path: Local checkpoint path for the task model.
        eval_parquet_path: Evaluation parquet path containing prompt text.
    """

    name: str
    model_path: Path
    eval_parquet_path: Path


@dataclass(frozen=True)
class RuntimeConfig:
    """Runtime and numerical settings for HVP-based curvature estimation.

    Args:
        output_root: Directory where CSV/JSON artifacts are written.
        device: Runtime device string (`cuda` or `cpu`).
        load_dtype: Weight loading dtype.
        max_prompt_tokens: Prompt truncation limit.
        max_new_tokens: Response length generated from model_t for NLL labels.
        batch_size: Dataloader batch size.
        max_batches: Number of batches used for Monte-Carlo curvature estimation.
        seed: Global random seed.
    """

    output_root: Path
    device: str
    load_dtype: torch.dtype
    max_prompt_tokens: int
    max_new_tokens: int
    batch_size: int
    max_batches: int
    seed: int


BASE_MODEL_ID = "Qwen/Qwen3-1.7B"
IF_MODEL_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface"
)
MATH_MODEL_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface"
)

# NOTE:
# - IF eval parquet exists in-repo by default.
# - For Math eval parquet, set the path to your prepared Math validation file before running.
IF_EVAL_PARQUET_PATH = Path("nemotron_evaluation/data/ifeval/val_verl_ready.parquet")
MATH_EVAL_PARQUET_PATH = Path("nemotron_evaluation/data/math/val_verl_ready.parquet")

IF_SPEC = TaskSpec(name="if", model_path=IF_MODEL_PATH, eval_parquet_path=IF_EVAL_PARQUET_PATH)
MATH_SPEC = TaskSpec(name="math", model_path=MATH_MODEL_PATH, eval_parquet_path=MATH_EVAL_PARQUET_PATH)

RUNTIME = RuntimeConfig(
    output_root=Path("merging_analysis/artifacts/hessian_task_vector_alignment"),
    device="cuda" if torch.cuda.is_available() else "cpu",
    load_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    max_prompt_tokens=1024,
    max_new_tokens=128,
    batch_size=1,
    max_batches=16,
    seed=7,
)


def set_seed(seed: int) -> None:
    """Set deterministic seeds for reproducible sampling and curvature estimates.

    Args:
        seed: Integer random seed.

    Returns:
        None.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_json_compatible(obj: Any) -> Any:
    """Recursively convert runtime objects into JSON-serializable primitives.

    Args:
        obj: Arbitrary nested object.

    Returns:
        JSON-compatible value.
    """

    if isinstance(obj, dict):
        return {str(key): to_json_compatible(value) for key, value in obj.items()}
    if isinstance(obj, list):
        return [to_json_compatible(value) for value in obj]
    if isinstance(obj, tuple):
        return [to_json_compatible(value) for value in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, torch.dtype):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


class PromptTextDataset(Dataset):
    """Dataset wrapper for prompt-text parquet rows.

    Args:
        prompts: Ordered list of prompt strings.

    Returns:
        Dataset that yields one prompt string per index.
    """

    def __init__(self, prompts: Sequence[str]) -> None:
        self.prompts = list(prompts)

    def __len__(self) -> int:
        return len(self.prompts)

    def __getitem__(self, index: int) -> str:
        return self.prompts[index]


def load_prompts_from_parquet(eval_parquet_path: Path) -> List[str]:
    """Load prompt text list from evaluation parquet with defensive column handling.

    Why this function exists:
        Different preprocessing pipelines store prompt text under different
        columns (`prompt_text`, `input`, or nested prompt fields). This loader
        centralizes fallback logic and fails loudly when no supported column exists.

    Args:
        eval_parquet_path: Source parquet path.

    Returns:
        List of prompt strings in source row order.

    Raises:
        FileNotFoundError: If parquet path does not exist.
        ValueError: If no recognized prompt column is found.
    """

    if not eval_parquet_path.exists():
        raise FileNotFoundError(f"Evaluation parquet not found: {eval_parquet_path}")

    dataframe = pd.read_parquet(eval_parquet_path)

    if "prompt_text" in dataframe.columns:
        return [str(value) for value in dataframe["prompt_text"].tolist()]

    if "input" in dataframe.columns:
        return [str(value) for value in dataframe["input"].tolist()]

    raise ValueError(
        "Evaluation parquet must contain one of {'prompt_text', 'input'} columns. "
        f"path={eval_parquet_path}, columns={list(dataframe.columns)}"
    )


def load_tokenizer_with_chat_template(model_name_or_path: str | Path) -> AutoTokenizer:
    """Load tokenizer with compatibility fallback for custom HF configs.

    Args:
        model_name_or_path: HF model id or local checkpoint path.

    Returns:
        Initialized tokenizer object.
    """

    resolved = str(model_name_or_path)
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            resolved,
            trust_remote_code=True,
            fix_mistral_regex=True,
        )
    except TypeError:
        tokenizer = AutoTokenizer.from_pretrained(resolved, trust_remote_code=True)

    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_causal_lm(model_name_or_path: str | Path, dtype: torch.dtype, device: str) -> AutoModelForCausalLM:
    """Load causal language model for analysis.

    Args:
        model_name_or_path: HF model id or local checkpoint path.
        dtype: Weight loading dtype.
        device: Target runtime device.

    Returns:
        Loaded model in evaluation mode on target device.
    """

    model = AutoModelForCausalLM.from_pretrained(
        str(model_name_or_path),
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        device_map=None,
    )
    model.to(device)
    model.eval()
    return model


def build_task_vector_dict(
    base_model: torch.nn.Module,
    task_model: torch.nn.Module,
) -> Dict[str, torch.Tensor]:
    """Build dense task vector dictionary `tau[name] = theta_task - theta_base`.

    Why this function exists:
        HVP directions must be aligned to exact model parameter names/shapes.
        This helper guarantees deterministic CPU float32 vectors for all
        floating parameters shared between base/task checkpoints.

    Args:
        base_model: Base anchor model.
        task_model: Task model whose direction is measured against base.

    Returns:
        Mapping `parameter_name -> task-vector tensor` on CPU float32.
    """

    base_named = dict(base_model.named_parameters())
    task_named = dict(task_model.named_parameters())

    task_vector: Dict[str, torch.Tensor] = {}

    with torch.no_grad():
        for parameter_name, base_parameter in tqdm(base_named.items(), desc="Build task vector"):
            if parameter_name not in task_named:
                continue

            task_parameter = task_named[parameter_name]

            if not torch.is_floating_point(base_parameter):
                continue

            if tuple(base_parameter.shape) != tuple(task_parameter.shape):
                raise ValueError(
                    f"Shape mismatch at {parameter_name}: "
                    f"base={tuple(base_parameter.shape)} task={tuple(task_parameter.shape)}"
                )

            # Use CPU float32 so direction vectors remain stable across devices.
            task_vector[parameter_name] = (
                task_parameter.detach().cpu().float() - base_parameter.detach().cpu().float()
            )

    if not task_vector:
        raise ValueError("Task vector is empty; check checkpoint compatibility")

    return task_vector


def align_params_and_direction(
    model_t: torch.nn.Module,
    task_vector: Mapping[str, torch.Tensor],
) -> Tuple[List[str], List[torch.nn.Parameter], List[torch.Tensor]]:
    """Align task-vector directions to model_t parameter order for autograd HVP.

    Args:
        model_t: Target task model where Hessian is evaluated.
        task_vector: Direction dictionary keyed by parameter name.

    Returns:
        Tuple `(parameter_names, parameters, directions)` with index alignment.
    """

    parameter_names: List[str] = []
    parameters: List[torch.nn.Parameter] = []
    directions: List[torch.Tensor] = []

    for parameter_name, parameter in model_t.named_parameters():
        if parameter_name not in task_vector:
            continue
        if not parameter.requires_grad:
            continue
        if not torch.is_floating_point(parameter):
            continue

        direction_tensor = task_vector[parameter_name].to(device=parameter.device, dtype=parameter.dtype)
        if tuple(direction_tensor.shape) != tuple(parameter.shape):
            raise ValueError(
                f"Direction shape mismatch at {parameter_name}: "
                f"direction={tuple(direction_tensor.shape)} parameter={tuple(parameter.shape)}"
            )

        parameter_names.append(parameter_name)
        parameters.append(parameter)
        directions.append(direction_tensor)

    if not parameters:
        raise ValueError("No aligned floating parameters found for HVP computation")

    return parameter_names, parameters, directions


def build_prompt_loader(prompts: Sequence[str], batch_size: int) -> DataLoader:
    """Build dataloader for prompt strings.

    Args:
        prompts: Prompt text list.
        batch_size: Batch size for iteration.

    Returns:
        Torch DataLoader that yields prompt batches.
    """

    return DataLoader(
        PromptTextDataset(prompts=prompts),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )


set_seed(RUNTIME.seed)
RUNTIME.output_root.mkdir(parents=True, exist_ok=True)

print("Configured Hessian-task-vector alignment study")
print(f"- Base model: {BASE_MODEL_ID}")
print(f"- IF model:   {IF_SPEC.model_path}")
print(f"- Math model: {MATH_SPEC.model_path}")
print(f"- Device:     {RUNTIME.device}")
print(f"- Max batches per estimate: {RUNTIME.max_batches}")
print(f"- Output root: {RUNTIME.output_root}")


In [ ]:
def build_sequence_log_prob_objective(
    model_t: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt_batch: Sequence[str],
    max_prompt_tokens: int,
    max_new_tokens: int,
) -> torch.Tensor:
    """Build differentiable sequence log-prob objective for one prompt batch.

    Why this objective is used:
        Fisher/Hessian-style curvature for language modeling can be estimated from
        generated continuations by maximizing `log p_theta(y|x)`. We sample `y`
        without gradients, then teacher-force the sampled tokens with gradients.

    Args:
        model_t: Task-t model used for curvature evaluation.
        tokenizer: Tokenizer compatible with model_t.
        prompt_batch: Batch of prompt strings.
        max_prompt_tokens: Prompt truncation limit.
        max_new_tokens: Number of generated response tokens.

    Returns:
        Scalar objective equal to sum of sequence log-probabilities over batch.

    Raises:
        ValueError: If all samples in batch produce empty continuations.
    """

    device = next(model_t.parameters()).device

    encoded_prompt = tokenizer(
        list(prompt_batch),
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_prompt_tokens,
    )

    input_ids = encoded_prompt["input_ids"].to(device)
    attention_mask = encoded_prompt["attention_mask"].to(device)

    # Sampling step intentionally excludes gradients to avoid expensive graph build.
    with torch.no_grad():
        generated_ids = model_t.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=1.0,
            temperature=1.0,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    batch_log_prob = torch.zeros((), dtype=torch.float32, device=device)
    valid_sequences = 0

    for row_index in range(generated_ids.shape[0]):
        prompt_len = int(attention_mask[row_index].sum().item())
        full_ids = generated_ids[row_index]
        full_len = int(full_ids.shape[0])

        if full_len <= prompt_len:
            # Skip degenerate sequence where generation ends immediately.
            continue

        valid_sequences += 1

        full_batch = full_ids.unsqueeze(0)
        full_attention = torch.ones_like(full_batch, device=device)
        logits = model_t(input_ids=full_batch, attention_mask=full_attention, use_cache=False).logits

        shifted_positions = torch.arange(prompt_len, full_len, device=device) - 1
        target_ids = full_ids[prompt_len:full_len]

        selected_logits = logits[0, shifted_positions, :].to(torch.float32)
        selected_log_probs = torch.log_softmax(selected_logits, dim=-1)

        sequence_log_prob = selected_log_probs.gather(
            dim=1,
            index=target_ids.unsqueeze(1),
        ).sum()

        batch_log_prob = batch_log_prob + sequence_log_prob

    if valid_sequences == 0:
        raise ValueError("All generated sequences were empty; cannot build curvature objective")

    # Normalize by valid sequence count to reduce batch-size sensitivity.
    return batch_log_prob / float(valid_sequences)


def hvp_quadratic_form(
    loss_objective: torch.Tensor,
    parameters: Sequence[torch.nn.Parameter],
    direction: Sequence[torch.Tensor],
) -> torch.Tensor:
    """Compute `v^T H v` via double autograd without forming full Hessian.

    Mathematical steps:
        1) `g = d(loss)/d(theta)`
        2) `H v = d(g)/d(theta) @ v`
        3) `v^T H v = sum_i <v_i, (H v)_i>`

    Args:
        loss_objective: Scalar differentiable objective.
        parameters: Parameter list aligned with `direction`.
        direction: Direction vector list (`tau_k`) aligned with parameters.

    Returns:
        Scalar tensor containing quadratic form `v^T H v`.
    """

    first_grads = torch.autograd.grad(
        loss_objective,
        parameters,
        create_graph=True,
        retain_graph=True,
        allow_unused=True,
    )

    used_parameters: List[torch.nn.Parameter] = []
    used_grads: List[torch.Tensor] = []
    used_direction: List[torch.Tensor] = []

    for parameter, grad_tensor, direction_tensor in zip(parameters, first_grads, direction):
        if grad_tensor is None:
            # Skip disconnected parameters to keep second autograd stable.
            continue
        used_parameters.append(parameter)
        used_grads.append(grad_tensor)
        used_direction.append(direction_tensor.detach())

    hvp_tensors = torch.autograd.grad(
        outputs=used_grads,
        inputs=used_parameters,
        grad_outputs=used_direction,
        create_graph=False,
        retain_graph=False,
        allow_unused=False,
    )

    quadratic_value = torch.zeros((), dtype=torch.float32, device=loss_objective.device)
    for direction_tensor, hvp_tensor in zip(used_direction, hvp_tensors):
        quadratic_value = quadratic_value + (direction_tensor * hvp_tensor).sum().to(torch.float32)

    return quadratic_value


def estimate_directional_curvature(
    model_t: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt_loader_t: DataLoader,
    parameters: Sequence[torch.nn.Parameter],
    direction: Sequence[torch.Tensor],
    max_batches: int,
    max_prompt_tokens: int,
    max_new_tokens: int,
    progress_desc: str,
) -> float:
    """Estimate expected directional curvature `E[v^T H_t v]` over prompt batches.

    Args:
        model_t: Task-t model whose Hessian is probed.
        tokenizer: Tokenizer for prompt encoding and generation.
        prompt_loader_t: Prompt dataloader for task-t validation prompts.
        parameters: Trainable parameter list aligned with direction.
        direction: Direction vector (`tau_k` or `tau_t`).
        max_batches: Maximum number of dataloader batches to process.
        max_prompt_tokens: Prompt truncation limit.
        max_new_tokens: Response generation length limit.
        progress_desc: TQDM progress label.

    Returns:
        Mean directional curvature over processed batches.

    Raises:
        RuntimeError: If no valid batch contributes to estimate.
    """

    model_t.eval()
    values: List[float] = []

    iterator = tqdm(prompt_loader_t, total=min(len(prompt_loader_t), max_batches), desc=progress_desc)
    for batch_index, prompt_batch in enumerate(iterator):
        if batch_index >= max_batches:
            break

        try:
            objective = build_sequence_log_prob_objective(
                model_t=model_t,
                tokenizer=tokenizer,
                prompt_batch=prompt_batch,
                max_prompt_tokens=max_prompt_tokens,
                max_new_tokens=max_new_tokens,
            )
            value = hvp_quadratic_form(
                loss_objective=objective,
                parameters=parameters,
                direction=direction,
            )
            values.append(float(value.detach().cpu().item()))
        except ValueError:
            # Skip batch when sampled continuations are all empty.
            continue
        finally:
            model_t.zero_grad(set_to_none=True)

            # Aggressive cleanup keeps repeated second-order passes stable.
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not values:
        raise RuntimeError("No valid curvature samples were collected")

    return float(sum(values) / len(values))


def compute_rho_stats(
    model_t: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt_loader_t: DataLoader,
    parameters_t: Sequence[torch.nn.Parameter],
    tau_t_direction: Sequence[torch.Tensor],
    tau_k_direction: Sequence[torch.Tensor],
    runtime: RuntimeConfig,
    target_task_name: str,
    comparison_task_name: str,
) -> Dict[str, Any]:
    """Compute normalized Hessian alignment metrics `rho(t,k)` and `|rho|`.

    Args:
        model_t: Task-t model where Hessian is evaluated.
        tokenizer: Tokenizer for objective construction.
        prompt_loader_t: Prompt dataloader for task-t data distribution.
        parameters_t: Parameter list aligned to direction lists.
        tau_t_direction: Self-task direction list.
        tau_k_direction: Cross-task direction list.
        runtime: Runtime config.
        target_task_name: Task name for `t`.
        comparison_task_name: Task name for `k`.

    Returns:
        Dictionary with self/cross curvature and rho metrics.
    """

    self_curvature = estimate_directional_curvature(
        model_t=model_t,
        tokenizer=tokenizer,
        prompt_loader_t=prompt_loader_t,
        parameters=parameters_t,
        direction=tau_t_direction,
        max_batches=runtime.max_batches,
        max_prompt_tokens=runtime.max_prompt_tokens,
        max_new_tokens=runtime.max_new_tokens,
        progress_desc=f"Self curvature ({target_task_name})",
    )

    cross_curvature = estimate_directional_curvature(
        model_t=model_t,
        tokenizer=tokenizer,
        prompt_loader_t=prompt_loader_t,
        parameters=parameters_t,
        direction=tau_k_direction,
        max_batches=runtime.max_batches,
        max_prompt_tokens=runtime.max_prompt_tokens,
        max_new_tokens=runtime.max_new_tokens,
        progress_desc=f"Cross curvature ({comparison_task_name} on {target_task_name})",
    )

    epsilon = 1e-12
    rho = float(cross_curvature / (self_curvature + epsilon))
    rho_abs = float(abs(cross_curvature) / (abs(self_curvature) + epsilon))

    return {
        "target_task": target_task_name,
        "comparison_task": comparison_task_name,
        "self_curvature": float(self_curvature),
        "cross_curvature": float(cross_curvature),
        "rho": float(rho),
        "rho_abs": float(rho_abs),
        "max_batches": int(runtime.max_batches),
    }


def save_results(output_root: Path, rows: Sequence[Mapping[str, Any]], runtime: RuntimeConfig) -> Dict[str, Path]:
    """Save Hessian alignment results to CSV and JSON summary files.

    Args:
        output_root: Destination directory for artifacts.
        rows: Per-direction result rows.
        runtime: Runtime configuration dataclass.

    Returns:
        Mapping from artifact label to saved path.
    """

    output_root.mkdir(parents=True, exist_ok=True)

    dataframe = pd.DataFrame(rows)
    csv_path = output_root / "rho_alignment_results.csv"
    json_path = output_root / "rho_alignment_summary.json"

    dataframe.to_csv(csv_path, index=False)

    summary_payload = {
        "created_at": datetime.now().isoformat(),
        "runtime": asdict(runtime),
        "rows": [dict(row) for row in rows],
        "independence_interpretation": "rho_abs near 0 implies near-flat cross-task curvature direction",
    }

    with json_path.open("w", encoding="utf-8") as file:
        json.dump(to_json_compatible(summary_payload), file, indent=2)

    return {
        "csv": csv_path,
        "json": json_path,
    }


# -----------------------------------------------------------------------------
# Step 1: Validate paths and load models/tokenizer.
# -----------------------------------------------------------------------------
for path_label, path_value in [
    ("IF model", IF_SPEC.model_path),
    ("Math model", MATH_SPEC.model_path),
    ("IF eval parquet", IF_SPEC.eval_parquet_path),
    ("Math eval parquet", MATH_SPEC.eval_parquet_path),
]:
    if not path_value.exists():
        raise FileNotFoundError(
            f"{path_label} path not found: {path_value}. Update config paths before running."
        )

tokenizer = load_tokenizer_with_chat_template(BASE_MODEL_ID)
base_model_cpu = load_causal_lm(BASE_MODEL_ID, dtype=RUNTIME.load_dtype, device="cpu")
if_model_cpu = load_causal_lm(IF_SPEC.model_path, dtype=RUNTIME.load_dtype, device="cpu")
math_model_cpu = load_causal_lm(MATH_SPEC.model_path, dtype=RUNTIME.load_dtype, device="cpu")

# -----------------------------------------------------------------------------
# Step 2: Build task vectors on CPU to avoid GPU memory spikes.
# -----------------------------------------------------------------------------
tau_if = build_task_vector_dict(base_model=base_model_cpu, task_model=if_model_cpu)
tau_math = build_task_vector_dict(base_model=base_model_cpu, task_model=math_model_cpu)

# Cleanup base model after vector extraction.
del base_model_cpu
gc.collect()

# -----------------------------------------------------------------------------
# Step 3: Build IF-task curvature probe (H_if).
# -----------------------------------------------------------------------------
if_model_t = load_causal_lm(IF_SPEC.model_path, dtype=RUNTIME.load_dtype, device=RUNTIME.device)
if_prompts = load_prompts_from_parquet(IF_SPEC.eval_parquet_path)
if_loader = build_prompt_loader(prompts=if_prompts, batch_size=RUNTIME.batch_size)

_, if_params, tau_if_on_if = align_params_and_direction(model_t=if_model_t, task_vector=tau_if)
_, _, tau_math_on_if = align_params_and_direction(model_t=if_model_t, task_vector=tau_math)

rho_if_math = compute_rho_stats(
    model_t=if_model_t,
    tokenizer=tokenizer,
    prompt_loader_t=if_loader,
    parameters_t=if_params,
    tau_t_direction=tau_if_on_if,
    tau_k_direction=tau_math_on_if,
    runtime=RUNTIME,
    target_task_name="if",
    comparison_task_name="math",
)

del if_model_t
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# -----------------------------------------------------------------------------
# Step 4: Build Math-task curvature probe (H_math).
# -----------------------------------------------------------------------------
math_model_t = load_causal_lm(MATH_SPEC.model_path, dtype=RUNTIME.load_dtype, device=RUNTIME.device)
math_prompts = load_prompts_from_parquet(MATH_SPEC.eval_parquet_path)
math_loader = build_prompt_loader(prompts=math_prompts, batch_size=RUNTIME.batch_size)

_, math_params, tau_math_on_math = align_params_and_direction(model_t=math_model_t, task_vector=tau_math)
_, _, tau_if_on_math = align_params_and_direction(model_t=math_model_t, task_vector=tau_if)

rho_math_if = compute_rho_stats(
    model_t=math_model_t,
    tokenizer=tokenizer,
    prompt_loader_t=math_loader,
    parameters_t=math_params,
    tau_t_direction=tau_math_on_math,
    tau_k_direction=tau_if_on_math,
    runtime=RUNTIME,
    target_task_name="math",
    comparison_task_name="if",
)

del math_model_t
del if_model_cpu
del math_model_cpu
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# -----------------------------------------------------------------------------
# Step 5: Save and display artifacts.
# -----------------------------------------------------------------------------
result_rows = [rho_if_math, rho_math_if]
artifact_paths = save_results(output_root=RUNTIME.output_root, rows=result_rows, runtime=RUNTIME)

display(pd.DataFrame(result_rows))
print("Saved artifacts:")
for artifact_name, artifact_path in artifact_paths.items():
    print(f"- {artifact_name}: {artifact_path}")

print("Interpretation hint: rho_abs closer to 0 means stronger second-order independence.")


## Output Artifacts

After execution, this notebook writes:

- `merging_analysis/artifacts/hessian_task_vector_alignment/rho_alignment_results.csv`
- `merging_analysis/artifacts/hessian_task_vector_alignment/rho_alignment_summary.json`

Interpretation:

- `rho_abs ~ 0`: cross-task direction is close to flat under target-task Hessian.
- `rho_abs ~ 1`: cross-task curvature magnitude is comparable to self-task curvature.

Before running, confirm `MATH_EVAL_PARQUET_PATH` exists and points to your Math validation parquet.
